# 06 - Ejercicios Prácticos: Taxonomía Farmacológica y Clustering Jerárquico de Principios Activos

## Curso de Machine Learning: Modelos No Supervisados

En este cuaderno pondrás en práctica el flujo completo de **Clustering Jerárquico Aglomerativo** aplicado a un problema de gran impacto en **Bioinformática, Química Médica y Diseño de Fármacos (*Drug Discovery*)**: la construcción de una taxonomía molecular y farmacocinética no supervisada sobre **45 principios activos y medicamentos**.

> **Nota:** Encontrarás las soluciones detalladas a estos ejercicios en la carpeta `.soluciones/06_Ejercicios_Practicos_Solucion.ipynb`. Te recomendamos encarecidamente resolver cada ejercicio por tu propia cuenta antes de consultar la solución.

---

### Instrucciones Generales
1. Ejecuta la celda de configuración inicial y generación del dataset molecular.
2. Lee cuidadosamente el contexto farmacológico de cada ejercicio.
3. Escribe tu código en las celdas vacías provistas (`# TU CODIGO AQUI`).
4. Anota tus observaciones e interpretaciones en las celdas de Markdown correspondientes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, cophenet
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## Contexto del Dataset: Propiedades Fisicoquímicas y Farmacocinéticas (45 Fármacos)

El departamento de farmacología computacional analiza 45 principios activos descritos por 7 propiedades moleculares (Regla de Lipinski / ADME):
- `peso_molecular_da`: Masa molecular en Daltons (Da).
- `logp_lipofilicidad`: Coeficiente de partición octanol/agua (mide afinidad grasa vs. agua).
- `donantes_enlace_h`: Número de átomos donantes de hidrógeno (OH, NH).
- `aceptores_enlace_h`: Número de átomos aceptores de hidrógeno (O, N).
- `superficie_polar_psa`: Área de superficie polar topológica en $\text{\AA}^2$.
- `biodisponibilidad_pct`: Fracción estimada del fármaco que llega inalterada a la circulación sistémica (%).
- `vida_media_h`: Tiempo en horas que tarda la concentración plasmática en reducirse al 50%.


In [ ]:
# Generador reproducible de 45 fármacos pertenecientes a 3 familias estructurales + análogos
def cargar_dataset_farmacos(seed=42):
    rng = np.random.default_rng(seed)
    # Familia 1: Antiinflamatorios y Analgésicos Pequeños Ácidos (n=16)
    f1 = np.column_stack([
        rng.normal(210.0, 25.0, 16).clip(150, 280),
        rng.normal(2.5, 0.6, 16).clip(1.2, 3.8),
        rng.normal(1.2, 0.4, 16).clip(1, 2),
        rng.normal(3.1, 0.7, 16).clip(2, 5),
        rng.normal(46.0, 8.0, 16).clip(30, 65),
        rng.normal(88.0, 6.0, 16).clip(75, 98),
        rng.normal(3.5, 1.2, 16).clip(1.5, 6.5)
    ])
    # Familia 2: Antibióticos Macrólidos / Lipopeptídicos Complejos (n=14)
    f2 = np.column_stack([
        rng.normal(740.0, 60.0, 14).clip(620, 860),
        rng.normal(1.8, 0.8, 14).clip(0.2, 3.2),
        rng.normal(4.8, 0.9, 14).clip(3, 7),
        rng.normal(13.5, 1.5, 14).clip(10, 17),
        rng.normal(175.0, 22.0, 14).clip(135, 230),
        rng.normal(38.0, 9.0, 14).clip(20, 55),
        rng.normal(45.0, 12.0, 14).clip(22, 70)
    ])
    # Familia 3: Agentes Cardiovasculares / Bloqueadores Lipofílicos (n=15)
    f3 = np.column_stack([
        rng.normal(360.0, 35.0, 15).clip(300, 440),
        rng.normal(4.6, 0.6, 15).clip(3.4, 5.8),
        rng.normal(1.1, 0.3, 15).clip(0, 2),
        rng.normal(4.5, 0.8, 15).clip(3, 6),
        rng.normal(58.0, 10.0, 15).clip(38, 80),
        rng.normal(64.0, 11.0, 15).clip(42, 85),
        rng.normal(18.5, 4.5, 15).clip(10, 28)
    ])
    X = np.vstack([f1, f2, f3])
    perm = rng.permutation(len(X))
    cols = ['peso_molecular_da', 'logp_lipofilicidad', 'donantes_enlace_h',
            'aceptores_enlace_h', 'superficie_polar_psa', 'biodisponibilidad_pct', 'vida_media_h']
    codigos = [f'DRUG_{i:02d}' for i in range(1, len(X) + 1)]
    return pd.DataFrame(X[perm], index=codigos, columns=cols)

df_farmacos = cargar_dataset_farmacos()
print(f'Dimensiones del dataset de principios activos: {df_farmacos.shape}')
df_farmacos.head()


---
## Ejercicio 1: Estandarización y Matriz de Disimilaridades

1. Estandariza la matriz de propiedades fisicoquímicas utilizando `StandardScaler` y guárdala en `X_farm_scaled`.
2. Calcula la matriz completa de distancias euclidianas por pares utilizando `pdist` y `squareform`.
3. **Pregunta:** ¿Por qué es indispensable estandarizar antes de construir el árbol jerárquico cuando se combinan Daltons con recuentos atómicos y porcentajes?


In [ ]:
# TU CODIGO AQUI: Estandarización y cálculo de distancias por pares



**Tus Observaciones (Ejercicio 1):**
*Escribe aquí tu respuesta...*


---
## Ejercicio 2: Selección del Criterio de Enlace Óptimo vía CPCC

1. Itera sobre los cuatro criterios de enlace clásicos: `['single', 'complete', 'average', 'ward']`.
2. Para cada enlace, construye la matriz de unión con `scipy.cluster.hierarchy.linkage` y calcula su **Coeficiente de Correlación Cofenética (CPCC)** con `cophenet`.
3. Genera una tabla comparativa y determina qué criterio preserva con mayor fidelidad la topología original.


In [ ]:
# TU CODIGO AQUI: Bucle de evaluación de enlaces y cálculo de CPCC



**Tus Observaciones (Ejercicio 2):**
*Escribe aquí qué enlace maximiza el CPCC y por qué...*


---
## Ejercicio 3: Dendrograma con Línea de Corte Horizontal

1. Grafica el dendrograma completo utilizando el enlace de `ward` con las etiquetas de los fármacos (`df_farmacos.index`).
2. Identifica el salto vertical más amplio y traza una línea horizontal punteada (`plt.axhline`) a la altura de corte adecuada.
3. Extrae las etiquetas de los clusters utilizando `fcluster` con `criterion='distance'` y almacénalas en la columna `df_farmacos['Cluster_Jerarquico']`.


In [ ]:
# TU CODIGO AQUI: Dendrograma visual y corte con fcluster



---
## Ejercicio 4: Clustermap Molecular Bidireccional

1. Genera un mapa de calor jerárquico bidireccional (`sns.clustermap`) sobre el DataFrame estandarizado.
2. Colorea el eje de las filas con las etiquetas de los clusters extraídos en el Ejercicio 3.
3. **Pregunta:** ¿Qué variables se agrupan juntas en el dendrograma superior y qué bloques moleculares caracterizan a cada grupo de fármacos?


In [ ]:
# TU CODIGO AQUI: Clustermap con sns.clustermap



---
## Ejercicio 5: Reglas Taxonómicas y Perfilamiento de Familias de Fármacos

1. Calcula la tabla de promedios de cada cluster en las **unidades moleculares reales**.
2. Entrena un `DecisionTreeClassifier(max_depth=2, random_state=42)` para extraer las reglas simples `IF-THEN` que diferencian las familias farmacológicas descubiertas.
3. Bautiza cada cluster de fármacos según su perfil (ej. analgésicos pequeños, macrólidos pesados, cardiovasculares lipofílicos).


In [ ]:
# TU CODIGO AQUI: Tabla de promedios originales y árbol sustituto



**Tus Conclusiones y Taxonomía Molecular (Ejercicio 5):**
- **Familia 1:** *...*
- **Familia 2:** *...*
- **Familia 3:** *...*
